In [ ]:
import polars as pl
import pickle
import torch
import tensorflow as tf
import os


In [ ]:
split_folder = "../data/Sports/GTS-q09-val_last_train_item-target_last-no_cold_items/"
path_to_embeddings =  "../data/Sports/embeddings.parquet"
out_folder = "../data/Sports/"
preproc_path = "../data/Sports/preprocessed.csv"

In [ ]:
val_input = pl.read_csv(f"{split_folder}/validation_input.csv")
val_target = pl.read_csv(f"{split_folder}/validation_target.csv")
train_input_and_target = pl.read_csv(f"{split_folder}/train.csv")
test_input = pl.read_csv(f"{split_folder}/test_input.csv")
test_target = pl.read_csv(f"{split_folder}/test_target.csv")

In [ ]:
def train_to_sasrec(df, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df = df.sort(["user_id", "timestamp"])
    df.select(["user_id", "item_id"]).write_csv(
        out_path,
        separator=" ",
        include_header=False,
    )


train_to_sasrec(train_input_and_target, f"{out_folder}/sasrec/sasrec_inter.txt")

In [ ]:
def create_emb_dict(path):
    df = pl.read_parquet(path)

    emb_dict = {
        int(item_id): [float(x) for x in emb]
        for item_id, emb in zip(df["item_id"], df["embedding"])
    }

    emb_size = len(next(iter(emb_dict.values())))
    emb_dict[0] = [1.0/emb_size] * emb_size
    return emb_dict



emb_dict = create_emb_dict(path_to_embeddings)

os.makedirs(f"{out_folder}/sasrec", exist_ok=True)
os.makedirs(f"{out_folder}/grid", exist_ok=True)

with open(f"{out_folder}/sasrec/embeddings_dict.pkl", "wb") as f:
    pickle.dump(emb_dict, f)


torch_pre_tesnor = [emb_dict[item_id] for item_id in range(1,len(emb_dict))]
torch_tensor = torch.tensor(torch_pre_tesnor)


torch.save(torch_tensor.cpu(), f"{out_folder}/grid/raw_item_embed_tensor.pt")

In [ ]:
def _long_to_sequences(df: pl.DataFrame) -> pl.DataFrame:
    """(user_id, item_id, timestamp)* -> one row per user with list[item_id]."""
    return (
        df.sort(["user_id", "timestamp"])
        .group_by("user_id", maintain_order=True)
        .agg((pl.col("item_id") - 1).alias("sequence_data"))
    )
def _write_partition(path: str, users: pl.DataFrame) -> None:
  options = tf.io.TFRecordOptions(compression_type="GZIP")
  with tf.io.TFRecordWriter(path, options=options) as writer:
    for row in users.iter_rows(named=True):
      uid = int(row["user_id"])
      seq = [int(x) for x in row["sequence_data"]]
      example = tf.train.Example(
        features=tf.train.Features(
          feature={
            "user_id": tf.train.Feature(
              int64_list=tf.train.Int64List(value=[uid])
            ),
            "sequence_data": tf.train.Feature(
              int64_list=tf.train.Int64List(value=seq)
            ),
          }
        )
      )
      writer.write(example.SerializeToString())
def export_grid_sequences(
    *,
    train: pl.DataFrame,
    val_input: pl.DataFrame,
    val_target: pl.DataFrame,
    test_input: pl.DataFrame,
    test_target: pl.DataFrame,
    out_folder: str,
    num_partitions: int = 175,
) -> None:
  splits = {
    "training": _long_to_sequences(train),
    "evaluation": _long_to_sequences(
      pl.concat([val_input, val_target], how="vertical")
    ),
    "testing": _long_to_sequences(
      pl.concat([test_input, test_target], how="vertical")
    ),
  }
  for split_name, seq_df in splits.items():
    split_dir = os.path.join(out_folder, split_name)
    os.makedirs(split_dir, exist_ok=True)
    seq_df = seq_df.sort("user_id")
    n_users = seq_df.height
    chunk_size = (n_users + num_partitions - 1) // num_partitions
    part_idx = 0
    for start in range(0, n_users, chunk_size):
      part = seq_df.slice(start, chunk_size)
      if part.is_empty():
        continue
      out_path = os.path.join(split_dir, f"partition_{part_idx}.tfrecord.gz")
      _write_partition(out_path, part)
      part_idx += 1

export_grid_sequences(
    train=train_input_and_target,
    val_input=val_input,
    val_target=val_target,
    test_input=test_input,
    test_target=test_target,
    out_folder=f"{out_folder}/grid/data", 
    num_partitions=32,
)


In [ ]:
def export_items_id_only(
    csv_path: str,
    out_folder: str,
    *,
    id_column: str = "item_id",
    records_per_file: int = 1024,
    unique: bool = True,
) -> None:
    df = pl.read_csv(csv_path)
    if id_column not in df.columns:
        raise ValueError(f"Column {id_column!r} not in {df.columns}")
    ids = df[id_column]
    if unique:
        ids = ids.unique().sort()
    else:
        ids = ids.sort()
    ids_0based = (ids - 1).cast(pl.Int64).to_list()
    items_dir = os.path.join(out_folder, "items")
    os.makedirs(items_dir, exist_ok=True)
    options = tf.io.TFRecordOptions(compression_type="GZIP")
    file_idx = 0
    for start in range(0, len(ids_0based), records_per_file):
        chunk = ids_0based[start : start + records_per_file]
        out_path = os.path.join(items_dir, f"data_{file_idx}.tfrecord.gz")
        with tf.io.TFRecordWriter(out_path, options=options) as writer:
            for item_id in chunk:
                example = tf.train.Example(
                    features=tf.train.Features(
                        feature={
                            "id": tf.train.Feature(
                                int64_list=tf.train.Int64List(value=[int(item_id)])
                            ),
                        }
                    )
                )
                writer.write(example.SerializeToString())
        file_idx += 1

export_items_id_only(csv_path=preproc_path, out_folder=f"{out_folder}/grid/data/")